
# Klike Data Science Challenge
## Parte 3 — Recommendations Engine

**Empresa:** Millai | Klike  
**Dataset:** `klike_challenge_dataset.csv`  
**Linguagem:** Python  
**Objetivo:** desenvolver um sistema de recomendações acionáveis e contextualizadas para otimização de criativos publicitários.

---

### Contextualização

Na Parte 1, foram investigadas as características das campanhas, a qualidade dos dados e as associações entre atributos dos criativos e métricas de desempenho.

Na Parte 2, foi desenvolvido um modelo de regressão Ridge reduzido, utilizando 10 características disponíveis antes da veiculação das campanhas para estimar o `klike_score`.

O modelo apresentou MAE de 6,685 e R² de 0,667 no conjunto de teste.

Nesta terceira etapa, o objetivo é transformar os resultados exploratórios e preditivos em um **Recommendations Engine**, capaz de analisar uma campanha e identificar oportunidades de melhoria.

O sistema utilizará uma abordagem híbrida, combinando:

- **Modelagem preditiva:** simulação de alterações nas características dos criativos por meio do Ridge reduzido.
- **Evidências estatísticas:** comparação de campanhas com características semelhantes, considerando seu contexto.
- **Priorização:** ordenação das sugestões conforme as diferenças previstas no Klike Score, respeitando a suficiência das evidências disponíveis.

As recomendações serão interpretadas como oportunidades de otimização, e não como garantias de melhoria. Diferenças preditivas e associações observacionais não constituem evidências de causalidade.



### Metodologia do Recommendations Engine

O desenvolvimento será organizado nas seguintes etapas:

1. **Preparação e reprodutibilidade:** carregamento do dataset e recuperação do Ridge reduzido desenvolvido na Parte 2.
2. **Validação das entradas:** identificação das características necessárias para analisar uma campanha e verificação de sua consistência.
3. **Geração de cenários alternativos:** definição de alterações acionáveis nas características dos criativos.
4. **Simulação preditiva:** utilização do Ridge para estimar diferenças no Klike Score entre a campanha original e os cenários alternativos.
5. **Análise contextual:** investigação das associações observadas entre características criativas e desempenho em campanhas comparáveis.
6. **Seleção e priorização:** filtragem de alterações não sustentadas pelas evidências e ordenação das oportunidades identificadas.
7. **Geração das recomendações:** apresentação de sugestões específicas, quantificadas e acompanhadas de suas limitações.
8. **Demonstração e validação:** execução do sistema em campanhas do dataset e verificação da consistência dos resultados.

**Princípios metodológicos:**

- Reutilizar o modelo selecionado na Parte 2, evitando treinamento desnecessário.
- Utilizar exclusivamente os registros de treinamento para construir as evidências estatísticas que fundamentam as recomendações.
- Preservar o conjunto de teste, evitando utilizá-lo na definição ou otimização das regras do engine.
- Não utilizar métricas pós-veiculação como entradas do modelo preditivo.
- Diferenciar explicitamente associações estatísticas, diferenças entre previsões e efeitos causais.
- Evitar recomendações baseadas em grupos com suporte amostral insuficiente.
- Não apresentar melhorias quantitativas sem uma estimativa calculada e devidamente contextualizada.

O sistema será desenvolvido inicialmente como um protótipo funcional em Python, utilizando funções independentes e reproduzíveis.



---

## 1. Preparação do ambiente e recuperação do modelo

### 1.1. Importação das bibliotecas e verificação dos arquivos

Nesta etapa, serão importadas as bibliotecas necessárias para manipulação dos dados, recuperação do modelo e reprodução da divisão entre treinamento e teste.

O Ridge reduzido foi salvo na Parte 2 utilizando `joblib`, juntamente com a lista ordenada das features utilizadas no treinamento.

Antes de carregar os arquivos, será verificada sua disponibilidade no diretório do projeto.

Essa abordagem permite reutilizar o pipeline previamente treinado sem realizar novos ajustes nos parâmetros do modelo.


In [1]:

from pathlib import Path

import joblib
import pandas as pd

from IPython.display import display
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge

RANDOM_STATE = 42

# Caminhos dos arquivos
DATA_PATH = Path("klike_challenge_dataset.csv")
MODEL_PATH = Path("models/ridge_klike.joblib")
FEATURES_PATH = Path("models/features_klike.joblib")

# Verificação de disponibilidade
arquivos = {
    "Dataset original": DATA_PATH,
    "Ridge reduzido": MODEL_PATH,
    "Features do modelo": FEATURES_PATH
}

status_arquivos = pd.DataFrame([
    {
        "Arquivo": nome,
        "Caminho": str(caminho),
        "Disponível": caminho.is_file()
    }
    for nome, caminho in arquivos.items()
])

display(status_arquivos)


,Arquivo,Caminho,Disponível
0,Dataset original,klike_challenge_dataset.csv,True
1,Ridge reduzido,models\ridge_klike.joblib,True
2,Features do modelo,models\features_klike.joblib,True



### 1.2. Carregamento dos dados e reconstrução da divisão treino/teste

O dataset será carregado sem alterações nos registros originais.

Em seguida, serão recuperados o pipeline Ridge treinado e a lista de features utilizada na Parte 2.

Para preservar a reprodutibilidade, será reconstruída a mesma divisão aleatória entre treinamento e teste, utilizando `test_size=0.20` e `random_state=42`.

As variáveis booleanas serão convertidas para a representação numérica adotada anteriormente, preservando os valores ausentes para tratamento pelo próprio pipeline.

A reconstrução da divisão é necessária porque as estatísticas contextuais do Recommendations Engine serão calculadas exclusivamente sobre as campanhas de treinamento.

O conjunto de teste permanecerá separado e não será utilizado na elaboração das recomendações.


In [2]:

# Verificar os arquivos necessários
arquivos_ausentes = [
    str(caminho)
    for caminho in arquivos.values()
    if not caminho.is_file()
]

if arquivos_ausentes:
    raise FileNotFoundError(
        f"Arquivos não encontrados: {arquivos_ausentes}"
    )

# Carregar dataset e artefatos da Parte 2
df = pd.read_csv(DATA_PATH)

modelo_ridge = joblib.load(MODEL_PATH)
features_reduzidas = joblib.load(FEATURES_PATH)

# Features definidas na Parte 2
features_esperadas = [
    "platform",
    "category",
    "is_retargeting",
    "video_duration_s",
    "format",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face",
    "text_density"
]

assert features_reduzidas == features_esperadas, (
    "As features carregadas diferem da Parte 2."
)

assert isinstance(modelo_ridge, Pipeline), (
    "O modelo carregado não é um Pipeline."
)

assert isinstance(modelo_ridge.named_steps["model"], Ridge), (
    "O estimador carregado não é um Ridge."
)

assert modelo_ridge.named_steps["model"].alpha == 1.0, (
    "O alpha do Ridge difere da Parte 2."
)

assert df["klike_score"].notna().all(), (
    "O dataset contém valores ausentes no target."
)

assert df["campaign_id"].is_unique, (
    "Existem identificadores de campanha duplicados."
)

# Preparar as mesmas entradas do Ridge reduzido
X = df[features_reduzidas].copy()
y = df["klike_score"].copy()

colunas_booleanas = [
    "is_retargeting",
    "has_subtitle",
    "has_cta",
    "has_hook",
    "has_face"
]

for coluna in colunas_booleanas:
    X[coluna] = X[coluna].map({
        True: 1,
        False: 0
    }).astype(float)

# Reproduzir a divisão utilizada na Parte 2
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=RANDOM_STATE
)

# Registros originais de treinamento para análises contextuais
df_train = df.loc[X_train.index].copy()

# Resumo da preparação
print("Modelo:", type(modelo_ridge.named_steps["model"]).__name__)
print("Alpha:", modelo_ridge.named_steps["model"].alpha)
print("Etapas do pipeline:", list(modelo_ridge.named_steps.keys()))

print("\nDimensões:")
print(f"Dataset original: {df.shape}")
print(f"Treinamento: {X_train.shape}")
print(f"Teste: {X_test.shape}")
print(f"Features utilizadas: {len(features_reduzidas)}")

print("\nFeatures do modelo:")
for i, feature in enumerate(features_reduzidas, start=1):
    print(f"{i:2d}. {feature}")

print("\nValidação dos índices:")
print(
    "Interseção treino/teste:",
    len(set(X_train.index) & set(X_test.index))
)
print(
    "Registros para evidência contextual:",
    len(df_train)
)


Modelo: Ridge
Alpha: 1.0
Etapas do pipeline: ['preprocessor', 'model']

Dimensões:
Dataset original: (500, 26)
Treinamento: (400, 10)
Teste: (100, 10)
Features utilizadas: 10

Features do modelo:
 1. platform
 2. category
 3. is_retargeting
 4. video_duration_s
 5. format
 6. has_subtitle
 7. has_cta
 8. has_hook
 9. has_face
10. text_density

Validação dos índices:
Interseção treino/teste: 0
Registros para evidência contextual: 400
